# US Tornado Data: Getting Started

Load and inspect the **2010–2025 v1.0.0** collection from NOAA and the U.S. Census Bureau.
This notebook uses the attached [Kaggle dataset, version 2](https://www.kaggle.com/datasets/jakevanslyke/us-tornado-data-2010-2025/versions/2).
It runs on CPU without internet, reads tables directly from `release.zip.bin`, and does not extract or modify the source collection.

The five sources are **SPC tornado tracks**, **NCEI Storm Events**, **NWS DAT damage surveys**, **Census county population/housing estimates**, and a **simplified 2020 Census county map**.
Their rows represent different things; do not add their counts to estimate a tornado total.

This is an inspection example, not a trained classifier or a joined event-level dataset.

## Locate and verify the frozen release

Kaggle mounts the attached dataset under `/kaggle/input`. The extra `.bin` suffix prevents Kaggle from unpacking the original compressed source files; Python's standard `zipfile` library still reads it normally.

The archive and manifest hashes below come from the [published release receipt](https://github.com/jakeryderv/us-tornado-data-2010-2025/blob/main/release/v1.0.0.json). If a future dataset version is attached, this check deliberately fails instead of silently changing the analysis. To run locally, set `TORNADO_RELEASE_ARCHIVE` to your downloaded `release.zip.bin`.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ARCHIVE_SHA256 = "237133bfec413fdab4fbda3bbf8080d198b935e9c6a182e2224e0f28d6666af0"
MANIFEST_SHA256 = "765f7b4402ed1e6af8e40e1d2665f5bd9f1901e31a6d1643cd64d6119ac51b9e"
configured = os.environ.get("TORNADO_RELEASE_ARCHIVE")
candidates = [Path(configured)] if configured else list(Path("/kaggle/input").rglob("release.zip.bin"))
if len(candidates) != 1:
    raise RuntimeError("Attach the US Tornado Data dataset or set TORNADO_RELEASE_ARCHIVE to its release.zip.bin file.")
archive_path = candidates[0]
with archive_path.open("rb") as stream:
    actual = hashlib.file_digest(stream, "sha256").hexdigest()
if actual != ARCHIVE_SHA256:
    raise ValueError("Wrong release archive: attach Kaggle version 2 of the v1.0.0 release.")
with ZipFile(archive_path) as archive:
    manifest_bytes = archive.read("release_manifest.json")
if hashlib.sha256(manifest_bytes).hexdigest() != MANIFEST_SHA256:
    raise ValueError("Release manifest does not match the pinned snapshot.")
manifest = json.loads(manifest_bytes)

def read_json(name):
    with ZipFile(archive_path) as archive:
        return json.loads(archive.read(name))

def read_csv(name, **kwargs):
    with ZipFile(archive_path) as archive, archive.open(name) as stream:
        return pd.read_csv(stream, **kwargs)

print(f"Verified {manifest['version']}: {manifest['file_count']:,} shared files, "
      f"{manifest['payload_bytes']/1e6:.1f} MB before compression")
display(pd.Series(manifest["source_snapshots"], name="Collected at"))

## File inventory

The archive retains source records, derived tornado extracts, schemas, provenance sidecars, and collection verification. NCEI raw files contain **all event types**, while `ncei_storm_events/tornado/` contains the exact tornado-related extracts. DAT includes all date-matching survey categories and no photos.

In [ ]:
files = pd.DataFrame(manifest["files"])
files["source_folder"] = files["path"].str.split("/").str[0]
files.loc[~files["path"].str.contains("/"), "source_folder"] = "release metadata"
inventory = files.groupby("source_folder").agg(files=("path", "size"), bytes=("bytes", "sum"))
inventory["MB"] = inventory["bytes"] / 1e6
display(inventory[["files", "MB"]].round(2))

## SPC: tornado tracks and recorded EF ratings

SPC supplies the main historical track catalog. `mag=-9` means **unknown**, not EF0. The 2010–2025 window lies within the Enhanced Fujita era; 2010 is not the date that scale was introduced. Track endpoints only approximate a path.

The plots below describe source records and their recorded ratings; changes over time also reflect reporting and assessment practices.

In [ ]:
spc = read_csv("spc/tornadoes_2010_2025.csv")
assert len(spc) == 20164
known = spc["mag"].between(0, 5)
ratings = spc.loc[known, "mag"].astype(int).value_counts().reindex(range(6), fill_value=0)
ratings.index = [f"EF{i}" for i in ratings.index]
ratings.loc["Unknown"] = int((~known).sum())
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
spc.groupby("yr").size().plot.bar(ax=axes[0], color="#277e8e")
axes[0].set(title="SPC tracks by year", xlabel="Year", ylabel="Track records")
ratings.plot.bar(ax=axes[1], color=["#277e8e"]*6+["#999999"])
axes[1].set(title="Recorded EF rating (unknown retained)", xlabel="Rating", ylabel="Track records")
fig.tight_layout(); plt.show()
display(ratings.rename("records").to_frame())
display(spc[["yr", "om", "date", "st", "mag", "slat", "slon", "elat", "elon", "len", "wid"]].head())

## NCEI and DAT: complementary records, unequal coverage

NCEI details are event/county segments, with related fatalities and locations. They are not one-to-one matches to SPC tracks. DAT's survey points, lines, and polygons are also separate units, with uneven geographic and historical coverage. A low survey count does not mean few tornadoes occurred.

Do not require DAT presence when defining a representative SPC master catalog. Source identifiers need explicit matching; DAT `event_id` is not NCEI `EVENT_ID`.

In [ ]:
ncei_rows = []
for year in range(2010, 2026):
    counts = {table: len(read_csv(f"ncei_storm_events/tornado/{year}_{table}.csv", low_memory=False))
              for table in ["details", "fatalities", "locations"]}
    ncei_rows.append({"year": year, **counts})
ncei_counts = pd.DataFrame(ncei_rows).set_index("year")
display(ncei_counts)
print("NCEI tornado-related row totals:", ncei_counts.sum().to_dict())

survey_rows = []
for year in range(2010, 2026):
    counts = {layer: read_json(f"nws_dat/{year}/{layer}/index.json")["features"]
              for layer in ["points", "lines", "polygons"]}
    survey_rows.append({"year": year, **counts})
survey_counts = pd.DataFrame(survey_rows).set_index("year")
display(survey_counts)
survey_counts.plot(subplots=True, figsize=(10, 7), marker="o", legend=False,
                   title=["DAT survey points", "DAT survey lines", "DAT survey polygons"])
plt.tight_layout(); plt.show()

index = read_json("nws_dat/2025/points/index.json")
first_batch = read_json("nws_dat/2025/points/" + index["batches"][0]["file"])
properties = pd.DataFrame([feature["properties"] for feature in first_batch["features"]])
display(properties[[c for c in ["objectid", "event_id", "stormdate", "efscale"] if c in properties]].head())
print("This sample is one batch; each index lists every batch for its year and layer.")

## Census: county context and a display map

Read `county_fips` as a string to preserve leading zeros. Population and housing units are county totals, not counts of people or buildings struck. Estimates use vintage 2020 for 2010–2019 and vintage 2025 for 2020–2025.

The fixed 2020 map is generalized to 1:5,000,000 and is unsuitable for precise damage-path intersections. Newer Connecticut planning regions have codes absent from this map. Matching codes elsewhere do not establish unchanged boundaries.

In [ ]:
counties = read_csv("census_population/county_context_2010_2025.csv", dtype={"county_fips": str})
assert len(counties) == 50294
assert counties["county_fips"].str.fullmatch(r"\d{5}").all()
display(counties.head())
missing_map = counties.loc[~counties["map_2020_fips_present"], ["county_fips", "state_name", "county_name"]].drop_duplicates()
display(missing_map)

from matplotlib.collections import LineCollection
boundary = read_json("census_boundaries/counties_2020_5m.geojson")
lines = []
for feature in boundary["features"]:
    geometry = feature["geometry"]
    polygons = geometry["coordinates"] if geometry["type"] == "MultiPolygon" else [geometry["coordinates"]]
    for polygon in polygons:
        ring = polygon[0]
        if all(-126 <= x <= -66 and 24 <= y <= 50 for x, y, *_ in ring):
            lines.append(ring)
fig, ax = plt.subplots(figsize=(13, 7))
ax.add_collection(LineCollection(lines, colors="#c5ccd1", linewidths=.3))
conus = spc[spc["slon"].between(-126, -66) & spc["slat"].between(24, 50)]
ax.scatter(conus["slon"], conus["slat"], s=2, alpha=.4, color="#277e8e", linewidths=0)
ax.set(xlim=(-126, -66), ylim=(24, 50), xlabel="Longitude", ylabel="Latitude",
       title="Recorded SPC tornado start locations, 2010–2025 (contiguous U.S. view)")
ax.set_aspect(1.25); plt.show()

## Before modeling EF ratings

- Preserve unknown ratings and assess severe class imbalance: only eight SPC records have EF5 labels in this snapshot.
- Damage, casualties, survey wind estimates, and narratives can reveal the rating process and leak the target. Decide which features are available at the intended prediction time.
- Match sources explicitly and keep segments from the same tornado or related outbreak together when splitting data.
- County context is broad exposure context. It does not locate individual buildings, and survey coverage is not uniform.
- This collection does not contain radar or environmental inputs for forecasting intensity before damage occurs.

Credit NOAA/NWS SPC, NOAA NCEI, NWS DAT, and the U.S. Census Bureau. Government records and project code have distinct reuse notices; see `DATA_SOURCES.md` and `CODE_LICENSE.txt` in the archive.

[Source code and methodology](https://github.com/jakeryderv/us-tornado-data-2010-2025) · [Dataset card](https://github.com/jakeryderv/us-tornado-data-2010-2025/blob/main/docs/DATASET_CARD.md) · [Hugging Face mirror](https://huggingface.co/datasets/jakeryderv/us-tornado-data-2010-2025)